# 1 - Config & Import

## 1.1 - Config

In [36]:
# 🔧 Config import
import sys
import os

try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(project_root)

from Config.LoggerConfig import colored_logger

logger = colored_logger()
current_file = os.path.basename(__file__) if '__file__' in globals() else "Notebook"
logger.info(f"Logger initialized ({current_file})")

[2025-06-14 11:42:01] INFO - 1013015657.py - Logger initialized (Notebook)


## 1.2 - Import

In [37]:
# 📁 Project root path setup
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

# 📦 External libs
import pandas as pd
import plotly.graph_objects as go
import tensorflow as tf
from tensorflow.keras.metrics import AUC, Precision, Recall
import importlib
import datetime
from tensorflow.keras.callbacks import TensorBoard

# 📂 Internal modules
import Data.DataFetcher
import Data.Features
import Data.Labels
import Data.Preprocessing
import Models.LSTMModel

def reload_modules():
    importlib.reload(Data.DataFetcher)
    importlib.reload(Data.Features)
    importlib.reload(Data.Labels)
    importlib.reload(Data.Preprocessing)
    importlib.reload(Models.LSTMModel)

[2025-06-14 11:42:25] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-14 11:42:25] INFO - Features.py - Logger initialized (Features.py)
[2025-06-14 11:42:25] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-14 11:42:25] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-14 11:42:25] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)


# 2 - Data

## 2.1 - Data fetcher

### Variables

In [111]:
 #📊 Data parameters
symbol = "EURUSD"
start_date = "12-06-2025"
end_date = "13-06-2025"
interval = "1s"

### Process

In [104]:
from ib_insync import *
from datetime import datetime, timedelta
import pandas as pd
import asyncio
import random

ib = IB()

async def get_IB_FX_Data(symbol: str, start_date: datetime, end_date: datetime) -> pd.DataFrame:
    await ib.connectAsync('127.0.0.1', 4002, clientId=random.randint(100, 999))

    contract = Forex(symbol)
    data = []

    slice_duration = timedelta(minutes=1)
    total_requests = int((end_date - start_date) / slice_duration)
    print(f"Total requests to perform: {total_requests}")

    current_start = start_date

    for i in range(total_requests):
        current_end = min(current_start + slice_duration, end_date)

        try:
            ticks = await ib.reqHistoricalTicksAsync(
                contract,
                startDateTime=current_start,
                endDateTime=current_end,
                numberOfTicks=1000,
                whatToShow='BID_ASK',
                useRth=False
            )

            for tick in ticks:
                data.append({
                    'time': tick.time,
                    'bid': tick.priceBid,
                    'ask': tick.priceAsk,
                    'bidSize': tick.sizeBid,
                    'askSize': tick.sizeAsk
                })

        except Exception as e:
            print(f"Error fetching {symbol} from {current_start} to {current_end}: {e}")

        print(f"{i+1}/{total_requests}")
        current_start = current_end
        await asyncio.sleep(0.3)  # avoid pacing violation

    ib.disconnect()

    df = pd.DataFrame(data)
    return df


import nest_asyncio
nest_asyncio.apply()

start = datetime.utcnow() - timedelta(hours=1)
end = datetime.utcnow()
start_date = datetime.strptime(start_date, "%d-%m-%Y")
end_date = datetime.strptime(end_date, "%d-%m-%Y")
df = asyncio.run(get_IB_FX_Data(symbol, start, end))
print(df.head())


TypeError: strptime() argument 1 must be str, not datetime.datetime

In [112]:
reload_modules()

# 📥 Load data
fetcher = Data.DataFetcher.DataFetcher(symbol, start_date, end_date, interval)

#fetcher.get_binance_data()

#fetcher.save_to_csv(directory="./Data")

fetcher.load_from_csv_ib(directory="./Data")

fetcher.resample_to_1m()

[2025-06-14 15:26:57] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-14 15:26:58] INFO - Features.py - Logger initialized (Features.py)
[2025-06-14 15:26:58] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-14 15:26:58] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-14 15:26:58] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)
[2025-06-14 15:26:59] INFO - DataFetcher.py - 📥 Data loaded from: ./Data/EURUSD_12-06-2025_13-06-2025_1s.csv
[2025-06-14 15:27:00] INFO - DataFetcher.py - Resampling complete. Final shape: (1439, 7)


### Visualization

In [113]:
print("#-----------------------")
print(fetcher.raw_data.columns)
print("#-----------------------")
print(fetcher.raw_data.head())
print("#-----------------------")
print(fetcher.raw_data.shape)
print("#-----------------------")

size = 200
df = fetcher.raw_data.copy()[-size:]

# Create a candlestick chart using Plotly
fig = go.Figure(data=[
    go.Candlestick(
        x=df.index,  # Using the timestamp index directly
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
        name="Candlesticks"
    )
])

# Customize layout
fig.update_layout(
    title=f"OHLC Candlestick Chart (Last {size} Points)",
    xaxis_title="Time",
    yaxis_title="Price",
    xaxis_rangeslider_visible=False,
    template="plotly_dark",
    height=500,
    width=int(500 * 1.618)
)

# Show the chart
fig.show()

#-----------------------
Index(['Open', 'High', 'Low', 'Close', 'Volume', 'bidSize', 'askSize'], dtype='object')
#-----------------------
                               Open      High       Low     Close    Volume  \
time                                                                          
2025-06-11 21:59:00+00:00  1.148820  1.148820  1.148820  1.148820    2000.0   
2025-06-11 22:00:00+00:00  1.148830  1.149160  1.148820  1.149125  396000.0   
2025-06-11 22:01:00+00:00  1.149140  1.149250  1.149130  1.149230  934000.0   
2025-06-11 22:02:00+00:00  1.149225  1.149255  1.149225  1.149250  498580.0   
2025-06-11 22:03:00+00:00  1.149250  1.149350  1.149240  1.149345  765220.0   

                            bidSize   askSize  
time                                           
2025-06-11 21:59:00+00:00    1000.0    1000.0  
2025-06-11 22:00:00+00:00  229000.0  167000.0  
2025-06-11 22:01:00+00:00  527000.0  407000.0  
2025-06-11 22:02:00+00:00  262880.0  235700.0  
2025-06-11 22:03:00+

## 2.2 - Feature Data

### Variables

In [114]:
# Resample with vwap
resample_period = '5min'

# Pivot Points
pivot_left = 15
pivot_right = 15

# Volume Pivot Points
duration_min = 240
n_cross = 7
std_factor = 1.0

# Std
std_window = 10

# Mean
mean_window = 10

### Process

In [121]:
reload_modules()

# Step 0: Initialize Features Dataframe
Features_Data = Data.Features.Features(fetcher.raw_data)

# Step 1: Resample with VWAP (Liquidity)
Features_Data.resample_with_vwap(resample_period)

# Step 2: Market Sessions (Market Context)
Features_Data.market_sessions()

# Step 3: Pivot Points (Support/Resistance)
Features_Data.Pivot_Points(pivot_left, pivot_right)

# Step 4: Volume Pivot Points (Volume-based S/R)
Features_Data.Volume_Pivot_Points(duration_min, n_cross, std_factor)

# Step 5: Volume Delta (Momentum)
Features_Data.add_volume_delta()

# Step 6: Cumulative Volume Delta - CVD (Momentum)
Features_Data.add_cvd()

# Step 7: Order Book Imbalance - OBI (Pressure)
Features_Data.add_obi()

# Step 8: Price Change (Price Action)
Features_Data.add_price_change()

# Step 9: Reaction Ratio (Liquidity)
Features_Data.add_reaction_ratio()

# Step 10: Rolling Standard Deviation of Price (Volatility)
Features_Data.add_rolling_std_price(std_window)

# Step 11: Rolling Mean of CVD (Smoothed Flow)
Features_Data.add_rolling_mean_cvd(mean_window)

[2025-06-14 15:34:30] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-14 15:34:30] INFO - Features.py - Logger initialized (Features.py)
[2025-06-14 15:34:30] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-14 15:34:30] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-14 15:34:30] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)
[2025-06-14 15:34:30] INFO - Features.py - Starting resampling with period: 5min
[2025-06-14 15:34:30] INFO - Features.py - Basic OHLCV resampling done.
[2025-06-14 15:34:30] INFO - Features.py - VWAP calculation completed.
[2025-06-14 15:34:30] INFO - Features.py - Resampling successful. Resulting shape: (293, 6)
[2025-06-14 15:34:30] INFO - Features.py - Starting market session flag generation.
[2025-06-14 15:34:30] INFO - Features.py - Datetime index successfully converted and localized.
[2025-06-14 15:34:30] INFO - Features.py - Market session flags successfully added.
[2025-06-14 15:34:30] INFO

['Open', 'High', 'Low', 'Close', 'Volume', 'VWAP_5m', 'London_Open', 'NY_Open', 'HK_Open', 'Dif_Low_Pivot', 'Dif_High_Pivot', 'Low_Pivot', 'High_Pivot', 'Rolling_VWAP_240min', 'Volume_Low_Pivot', 'Volume_High_Pivot', 'Dif_Volume_Low_Pivot', 'Dif_Volume_High_Pivot']


KeyError: 'bidSize'

### Visualization

In [26]:
#-------------------------------------------------------------------------------

print("#----------------------------------------------------------------------")
print("Shape:", Features_Data.data.shape)
print("#----------------------------------------------------------------------")
print(Features_Data.data.columns)
print("#----------------------------------------------------------------------")
print(Features_Data.data.head())
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------
df = Features_Data.data.copy()[-300:]

# 1. Candlestick chart with VWAP_5m
fig1 = go.Figure()
fig1.add_trace(go.Candlestick(x=df.index,
                              open=df['Open'],
                              high=df['High'],
                              low=df['Low'],
                              close=df['Close'],
                              name="Candlestick"))
fig1.add_trace(go.Scatter(x=df.index, y=df['VWAP_5m'], mode='lines', name='VWAP_5m', line=dict(color='white')))
fig1.update_layout(title="Candlestick Chart with VWAP_5m", xaxis_title="Time", yaxis_title="Price", template="plotly_dark",xaxis_rangeslider_visible=False,)

# 2. Market session open status
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df.index, y=df["London_Open"], mode='lines', name="London"))
fig2.add_trace(go.Scatter(x=df.index, y=df["NY_Open"], mode='lines', name="New York"))
fig2.add_trace(go.Scatter(x=df.index, y=df["HK_Open"], mode='lines', name="Hong Kong"))
fig2.update_layout(title="Market Sessions", xaxis_title="Time", yaxis_title="Market Open (1=open)", template="plotly_dark")

# 3. Pivot points for price
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=df.index, y=df["Low"], mode='lines', name="Low", line=dict(width=0.8)))
fig3.add_trace(go.Scatter(x=df.index, y=df["Low_Pivot"], mode='lines', name="Low Pivot", line=dict(width=1.2, dash='dash')))
fig3.add_trace(go.Scatter(x=df.index, y=df["High"], mode='lines', name="High", line=dict(width=0.8)))
fig3.add_trace(go.Scatter(x=df.index, y=df["High_Pivot"], mode='lines', name="High Pivot", line=dict(width=1.2, dash='dash')))
fig3.update_layout(title="Price Pivot Points Detection", xaxis_title="Time", yaxis_title="Price", template="plotly_dark")

# 4. Volume pivot points
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=df.index, y=df["Volume_Low_Pivot"].tail(200), mode='lines', name="Volume Low Pivot", line=dict(width=1.2, dash='dash')))
fig4.add_trace(go.Scatter(x=df.index, y=df["VWAP_5m"].tail(200), mode='lines', name="VWAP 5m", line=dict(width=0.8)))
fig4.add_trace(go.Scatter(x=df.index, y=df["Rolling_VWAP_240min"].tail(200), mode='lines', name="Rolling VWAP 240min"))
fig4.add_trace(go.Scatter(x=df.index, y=df["Volume_High_Pivot"].tail(200), mode='lines', name="Volume High Pivot", line=dict(width=1.2, dash='dash')))
fig4.update_layout(title="Volume Pivot Points Detection", xaxis_title="Time", yaxis_title="Price", template="plotly_dark")

fig1.show()
fig2.show()
fig3.show()
fig4.show()

#----------------------------------------------------------------------
Shape: (32600, 19)
#----------------------------------------------------------------------
Index(['Open', 'High', 'Low', 'Close', 'Volume', 'VWAP_5m', 'London_Open',
       'NY_Open', 'HK_Open', 'Is_Noise', 'Dif_Low_Pivot', 'Dif_High_Pivot',
       'Low_Pivot', 'High_Pivot', 'Rolling_VWAP_240min', 'Volume_Low_Pivot',
       'Volume_High_Pivot', 'Dif_Volume_Low_Pivot', 'Dif_Volume_High_Pivot'],
      dtype='object')
#----------------------------------------------------------------------
                        Open     High      Low    Close   Volume  \
Timestamp                                                          
2024-12-31 23:00:00  93469.1  93550.9  93362.9  93394.1  484.937   
2024-12-31 23:05:00  93394.1  93623.1  93356.6  93623.1  433.904   
2024-12-31 23:10:00  93623.0  93736.9  93576.0  93687.3  268.385   
2024-12-31 23:15:00  93687.2  93689.7  93617.7  93631.1  140.948   
2024-12-31 23:20:00  93631.1 

## 2.3 - Label Data

### Variables

In [27]:
# Step 1: Categorize Pivot Points
look_forward=20

# Step 2: Categorize Volume Pivot Points
look_forward=20

### Process

In [28]:
# Step 0: Initialize Labels Dataframe
Labels_Data = Data.Labels.Labels(Features_Data.data)

# Step 1: Categorize Pivot Points
Labels_Data.Categorize_Pivot_Points(look_forward)

# Step 2: Categorize Volume Pivot Points
Labels_Data.Categorize_Volume_Pivot_Points(look_forward)

[2025-06-08 16:59:35] INFO - Labels.py - Starting Categorize_Pivot_Points with look_forward=20
[2025-06-08 16:59:51] INFO - Labels.py - Categorize_Pivot_Points completed successfully.
[2025-06-08 16:59:51] INFO - Labels.py - Starting Categorize_Volume_Pivot_Points with look_forward=20
[2025-06-08 17:00:05] INFO - Labels.py - Categorize_Volume_Pivot_Points completed successfully.


### Visualization

In [29]:
#-------------------------------------------------------------------------------

print("#----------------------------------------------------------------------")
print("Shape:", Labels_Data.data.shape)
print("#----------------------------------------------------------------------")
print(Labels_Data.data.columns)
print("#----------------------------------------------------------------------")
print(Labels_Data.data.head())
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

df = Labels_Data.data.copy()[-300:]

# Fréquences des 0 et 1 pour les colonnes cibles
frequency = df[["Low_Below_Pivot", "High_Above_Pivot"]].apply(lambda col: col.value_counts()).fillna(0).astype(int)
print("\nLabel Frequency (0 = no breakout, 1 = breakout):")
print(frequency.head(3))

# Initialiser les couleurs
colors_combined = ['lightgrey'] * len(df)
for i in range(len(df)):
    if df.iloc[i]['High_Above_Pivot'] == 1:
        colors_combined[i] = 'green'
    elif df.iloc[i]['Low_Below_Pivot'] == 1:
        colors_combined[i] = 'red'

# Création du graphique
fig = go.Figure()

# Trace VWAP_5m avec couleur dynamique
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['VWAP_5m'],
    mode='markers+lines',
    marker=dict(color=colors_combined, size=6),
    name='VWAP_5m',
    line=dict(color='lightgrey')
))

# Trace Low_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Low_Pivot'],
    mode='lines',
    line=dict(color='red', width=1),
    name='Low_Pivot'
))

# Trace High_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['High_Pivot'],
    mode='lines',
    line=dict(color='green', width=1),
    name='High_Pivot'
))

# Mise en page
fig.update_layout(
    title='VWAP_5m with Pivot Breakout Coloring and Pivot Lines',
    xaxis_title='Time',
    yaxis_title='Price',
    template='plotly_dark',
    height=500
)

fig.show()
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

# Fréquences des 0 et 1 pour les colonnes VWAP
frequency = df[["VWAP_Below_Volume_Low", "VWAP_Above_Volume_High"]].apply(lambda col: col.value_counts()).fillna(0).astype(int)
print("\nLabel Frequency (0 = no breakout, 1 = breakout):")
print(frequency.head(3))

# Initialiser les couleurs
colors_combined = ['lightgrey'] * len(df)
for i in range(len(df)):
    if df.iloc[i]['VWAP_Above_Volume_High'] == 1:
        colors_combined[i] = 'green'
    elif df.iloc[i]['VWAP_Below_Volume_Low'] == 1:
        colors_combined[i] = 'red'

# Création du graphique
fig = go.Figure()

# Trace VWAP_5m avec couleur dynamique
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['VWAP_5m'],
    mode='markers+lines',
    marker=dict(color=colors_combined, size=6),
    name='VWAP_5m',
    line=dict(color='lightgrey')
))

# Trace Volume_Low_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Volume_Low_Pivot'],
    mode='lines',
    line=dict(color='red', width=1),
    name='Volume_Low_Pivot'
))

# Trace Volume_High_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Volume_High_Pivot'],
    mode='lines',
    line=dict(color='green', width=1),
    name='Volume_High_Pivot'
))

# Mise en page
fig.update_layout(
    title='VWAP_5m with Volume Pivot Breakout Coloring and Pivot Lines',
    xaxis_title='Time',
    yaxis_title='Price',
    template='plotly_dark',
    height=500
)

fig.show()

#-------------------------------------------------------------------------------

#----------------------------------------------------------------------
Shape: (32600, 23)
#----------------------------------------------------------------------
Index(['Open', 'High', 'Low', 'Close', 'Volume', 'VWAP_5m', 'London_Open',
       'NY_Open', 'HK_Open', 'Is_Noise', 'Dif_Low_Pivot', 'Dif_High_Pivot',
       'Low_Pivot', 'High_Pivot', 'Rolling_VWAP_240min', 'Volume_Low_Pivot',
       'Volume_High_Pivot', 'Dif_Volume_Low_Pivot', 'Dif_Volume_High_Pivot',
       'Low_Below_Pivot', 'High_Above_Pivot', 'VWAP_Below_Volume_Low',
       'VWAP_Above_Volume_High'],
      dtype='object')
#----------------------------------------------------------------------
                        Open     High      Low    Close   Volume  \
Timestamp                                                          
2024-12-31 23:00:00  93469.1  93550.9  93362.9  93394.1  484.937   
2024-12-31 23:05:00  93394.1  93623.1  93356.6  93623.1  433.904   
2024-12-31 23:10:00  93623.0  93736.9  93576.0  93687.3  268.

#----------------------------------------------------------------------

Label Frequency (0 = no breakout, 1 = breakout):
   VWAP_Below_Volume_Low  VWAP_Above_Volume_High
0                    255                     231
1                     45                      69


## 2.4  - Preprocessing

### Variables

In [33]:
# Step 0  Initialize Processing Data
print(Labels_Data.data.columns)
train_test_data = pd.DataFrame({"Feature_1": Labels_Data.data['VWAP_5m'],
                                "Feature_2": Labels_Data.data["London_Open"],
                                "Feature_3": Labels_Data.data["NY_Open"],
                                "Feature_4": Labels_Data.data["HK_Open"],
                                "Feature_5": Labels_Data.data["Is_Noise"],
                                "Feature_6": Labels_Data.data['Dif_Low_Pivot'],
                                "Feature_7": Labels_Data.data['Dif_High_Pivot'],
                                "Feature_8": Labels_Data.data['Dif_Volume_Low_Pivot'],
                                "Feature_9": Labels_Data.data['Dif_Volume_High_Pivot'],

                                "Label_1": Labels_Data.data['Low_Below_Pivot'],
                                "Label_2": Labels_Data.data['High_Above_Pivot'],
                                "Label_3": Labels_Data.data['VWAP_Below_Volume_Low'],
                                "Label_4": Labels_Data.data['VWAP_Above_Volume_High'],
                                }).dropna()

# Step 1: Create Train/Test Data
lookback = 50
size_test_prct = 0.3

# Step 2: Suggest BatchSize
feature_dim = 9
label_dim = 4
lookback = 50
reserved_ram_gb = 1.0
hidden_dim = [64, 64]

# Step 3: Create DataLoaders
val_ratio = 0.2

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'VWAP_5m', 'London_Open',
       'NY_Open', 'HK_Open', 'Is_Noise', 'Dif_Low_Pivot', 'Dif_High_Pivot',
       'Low_Pivot', 'High_Pivot', 'Rolling_VWAP_240min', 'Volume_Low_Pivot',
       'Volume_High_Pivot', 'Dif_Volume_Low_Pivot', 'Dif_Volume_High_Pivot',
       'Low_Below_Pivot', 'High_Above_Pivot', 'VWAP_Below_Volume_Low',
       'VWAP_Above_Volume_High'],
      dtype='object')


### Process

In [34]:
# Step 0  Initialize Processing Data
Preprocessing_Data = Data.Preprocessing.Preprocessing(train_test_data)

# Step 1: Create Train/Test Data
Preprocessing_Data.create_train_test_data(lookback,size_test_prct)

# Step 2: Suggest BatchSize
Preprocessing_Data.suggest_batch_size(feature_dim, label_dim, lookback, reserved_ram_gb, hidden_dim)

# Step 3: Create DataLoaders
Preprocessing_Data.create_dataloaders(val_ratio)

[2025-06-08 17:00:35] INFO - Preprocessing.py - Starting creation of train/test data.
[2025-06-08 17:00:35] INFO - Preprocessing.py - Selected 9 feature(s) and 4 label(s).
[2025-06-08 17:00:35] INFO - Preprocessing.py - Features normalized (min-max).
[2025-06-08 17:00:35] INFO - Preprocessing.py - Generated 29479 sequences of length 50.
[2025-06-08 17:00:35] INFO - Preprocessing.py - Train/Test split: 20635 train / 8844 test samples.
[2025-06-08 17:00:35] INFO - Preprocessing.py - ✅ Train/test tensors successfully created.
[2025-06-08 17:00:35] INFO - Preprocessing.py - Starting batch size estimation...
[2025-06-08 17:00:35] INFO - Preprocessing.py - Creating TensorFlow DataLoaders...
[2025-06-08 17:00:35] INFO - Preprocessing.py - Train/Val split: 16508 train / 4127 val samples
[2025-06-08 17:00:35] INFO - Preprocessing.py - ✅ TensorFlow DataLoaders created successfully.


### Verification

In [35]:
print("x_train shape:", Preprocessing_Data.x_train.shape)
print("y_train shape:", Preprocessing_Data.y_train.shape)
print("x_test shape:", Preprocessing_Data.x_test.shape)
print("y_test shape:", Preprocessing_Data.y_test.shape)

for row in Preprocessing_Data.report_lines:
    print(row)

for x_batch, y_batch in Preprocessing_Data.train_loader.take(1):
    print("Train batch shape:", x_batch.shape, y_batch.shape)

for x_batch, y_batch in Preprocessing_Data.val_loader.take(1):
    print("Val batch shape:", x_batch.shape, y_batch.shape)

x_train shape: (20635, 50, 9)
y_train shape: (20635, 4)
x_test shape: (8844, 50, 9)
y_test shape: (8844, 4)
--- Batch Size Estimation Report (TensorFlow) ---
Total RAM (GB):         7.78
Reserved RAM (GB):      1.0
Usable RAM (GB):        6.78
Samples (n_samples):    20635
Features per timestep:  9
Labels (n_labels):      4
Lookback (timesteps):   50
Hidden dimensions:      [64, 64] (layers: 2)
Bytes/sample:           51.77 KB
Max batch size (RAM):   137257.0
Max batch size (CPU):   64
Using GPU:              No
Final suggested batch:  64
--------------------------------------------------
Train batch shape: (64, 50, 9) (64, 4)
Val batch shape: (64, 50, 9) (64, 4)


# 3 - Model

## 3.1 - Hyperparameters

In [53]:
# =======================
# 🔧 Hyperparameters
# =======================
# Architecture
feature_dim             = 9
label_dim               = 4
lookback                = 50
hidden_dims             = [64, 64]              # List of LSTM hidden units
dropout                 = 0.3                   # Dropout after each LSTM layer
recurrent_dropout       = 0.0                   # Dropout on recurrent state (LSTM-specific)
activation              = 'tanh'                # Activation inside LSTM cell
recurrent_activation    = 'sigmoid'             # Activation for gates
kernel_initializer      = 'glorot_uniform'      # Init method for weights
use_bias                = True                  # Include bias in LSTM and Dense layers
bidirectional           = False                 # Use bidirectional LSTM
use_batch_norm          = False                 # Add BatchNormalization after LSTM
dense_units             = None                  # Optional intermediate Dense layer
activation_dense        = 'relu'                # Activation for Dense layer

# Regularization
kernel_regularizer_l2   = 1e-4                  # L2 regularization
recurrent_regularizer_l2= None                  # Optional
activity_regularizer_l2 = None

# Optimizer
learning_rate           = 0.001
clipnorm                = 1.0
beta_1                  = 0.9
beta_2                  = 0.999
epsilon                 = 1e-7

# Loss & Metrics
loss_function           = 'binary_crossentropy'
metrics_list            = [
    'binary_accuracy',
    tf.keras.metrics.AUC(name='auc'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    Models.LSTMModel.F1Score(name='f1_score')  # Custom metric
]

# Early stopping
early_stopping_patience = 3
early_stopping_min_delta= 1e-4
restore_best_weights    = True
monitor_metric          = 'val_loss'

# Reduce LR on Plateau
use_reduce_lr           = True
reduce_lr_patience      = 2
reduce_lr_factor        = 0.5
reduce_lr_min_lr        = 1e-6

# Checkpoint
use_model_checkpoint    = True
checkpoint_filepath     = 'best_model.keras'

# Training
batch_size              = 64
shuffle                 = True
normalize_input         = True
window_stride           = 1
epochs                  = 50

In [54]:
reload_modules()

model = Models.LSTMModel.LSTMModel(
    feature_dim=feature_dim,
    label_dim=label_dim,
    hidden_dims=hidden_dims,
    lookback=lookback,
    dropout=dropout,
    recurrent_dropout=recurrent_dropout,
    activation=activation,
    recurrent_activation=recurrent_activation,
    kernel_initializer=kernel_initializer,
    use_bias=use_bias,
    bidirectional=bidirectional,
    use_batch_norm=use_batch_norm,
    dense_units=dense_units,
    activation_dense=activation_dense,
    kernel_regularizer_l2=kernel_regularizer_l2,
    loss_function=loss_function,
    metrics_list=metrics_list,
    learning_rate=learning_rate,
    clipnorm=clipnorm,
    beta_1=beta_1,
    beta_2=beta_2,
    epsilon=epsilon,
    early_stopping_patience=early_stopping_patience,
    early_stopping_min_delta=early_stopping_min_delta,
    restore_best_weights=restore_best_weights,
    monitor_metric=monitor_metric,
    use_reduce_lr=use_reduce_lr,
    reduce_lr_patience=reduce_lr_patience,
    reduce_lr_factor=reduce_lr_factor,
    reduce_lr_min_lr=reduce_lr_min_lr,
    use_model_checkpoint=use_model_checkpoint,
    checkpoint_filepath=checkpoint_filepath
)

optimizer = model.optimizer()
model.compile()
model.build()
callbacks = model.get_callbacks()


[2025-06-08 17:39:46] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-08 17:39:46] INFO - Features.py - Logger initialized (Features.py)
[2025-06-08 17:39:46] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-08 17:39:46] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-08 17:39:46] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)


TypeError: LSTMModel.__init__() missing 17 required positional arguments: 'loss_function', 'metrics_list', 'learning_rate', 'clipnorm', 'beta_1', 'beta_2', 'epsilon', 'early_stopping_patience', 'early_stopping_min_delta', 'restore_best_weights', 'monitor_metric', 'use_reduce_lr', 'reduce_lr_patience', 'reduce_lr_factor', 'reduce_lr_min_lr', 'use_model_checkpoint', and 'checkpoint_filepath'

# 4 - Training & Visualization

In [38]:
# Définir le répertoire de logs avec un horodatage
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

# Créer le callback TensorBoard
tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1,
    write_graph=True,
    write_images=True,
    update_freq='epoch'
)

# -----------------------------------------------------------
# Entraîner le modèle avec TensorBoard et EarlyStopping
model.fit(
    Preprocessing_Data.train_loader,
    validation_data=Preprocessing_Data.val_loader,
    epochs=1,
    callbacks=[early_stop, tensorboard_callback]
)

258/258 ━━━━━━━━━━━━━━━━━━━━ 33s 81ms/step - auc: 0.5310 - binary_accuracy: 0.7478 - f1_score: 0.0446 - loss: 0.5730 - precision: 0.2272 - recall: 0.0318 - val_auc: 0.6138 - val_binary_accuracy: 0.7638 - val_f1_score: 0.0000e+00 - val_loss: 0.5400 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00


In [42]:
import webbrowser
webbrowser.open('http://localhost:6006')

True